In [1]:
import ee
import geemap
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import sys

In [3]:
# Authenticate and Initialize Earth Engine
ee.Authenticate()
ee.Initialize(project= "charrell-personal")

In [4]:
#Initialize Map
Map = geemap.Map()
Map = geemap.Map(height="800px")

In [5]:
from tgbs_rs.preprocessing import get_s2_sr_collection
from tgbs_rs.utils import (build_default_sites_featurecollection, get_sites_geometry)
from tgbs_rs.summaries import (collection_to_site_timeseries, build_monthly_band_composites)

#### Define User Inputs

In [6]:
# Build site feature collection
sites_fc = build_default_sites_featurecollection()

# Get merged geometry for filtering image collections
sites_geom = get_sites_geometry(sites_fc)


ks_rehab_geom = sites_fc.filter(ee.Filter.eq("site_id", "ks_rehab")).geometry()
buda_geom = sites_fc.filter(ee.Filter.eq("site_id", "buda")).geometry()
gogoni_geom = sites_fc.filter(ee.Filter.eq("site_id", "gogoni")).geometry()
shimba_hills_geom = sites_fc.filter(ee.Filter.eq("site_id", "shimba_hills")).geometry()

In [7]:
site_vis = {
    'color': 'red',
    'fillColor': '00000000',
    'width': 2
}

In [8]:
# Build processed Sentinel-2 collection
s2_col = get_s2_sr_collection(
    aoi=sites_geom,
    start_date=ee.Date("2020-01-01"),
    end_date=ee.Date("2024-12-31"),
    apply_water_masking=True,
)

In [9]:
# Build monthly NDVI composites
monthly_ndvi_col = build_monthly_band_composites(
    collection=s2_col,
    band="NDVI",
    start_date=ee.Date("2020-01-01"),
    end_date=ee.Date("2024-12-31"),
    composite_stat="median",
)

In [10]:
# Summarize monthly NDVI composites over all sites
ndvi_ts_fc = collection_to_site_timeseries(
    collection=monthly_ndvi_col,
    sites_fc=sites_fc,
    bands=["NDVI"],
    reducer=ee.Reducer.mean(),
    scale=10,
    tile_scale=4,
)


In [12]:

# Convert to dataframe
ndvi_ts_df = geemap.ee_to_df(ndvi_ts_fc)

Exception: Too many concurrent aggregations.

In [ ]:
def plot_site_timeseries(
    df,
    value_col,
    date_col="date",
    site_col="site_name",
    figsize=(12, 6),
    marker="o",
    linewidth=1.5,
):
    """
    Plot a time series for each site in a long-format DataFrame.

    Parameters
    ----------
    df : pandas.DataFrame
        Long-format dataframe containing one row per site-date observation.
    value_col : str
        Column to plot on the y-axis, e.g. 'NDVI'.
    date_col : str, default 'date'
        Name of the date column.
    site_col : str, default 'site_name'
        Name of the site label column.
    figsize : tuple, default (12, 6)
        Figure size.
    marker : str, default 'o'
        Marker style.
    linewidth : float, default 1.5
        Line width.

    Returns
    -------
    matplotlib.axes.Axes
        The plotted axes object.
    """
    plot_df = df.copy()

    plot_df[date_col] = pd.to_datetime(plot_df[date_col], errors="coerce")
    plot_df = plot_df.dropna(subset=[date_col, value_col, site_col])
    plot_df = plot_df.sort_values([site_col, date_col])

    fig, ax = plt.subplots(figsize=figsize)

    for site_name, group in plot_df.groupby(site_col):
        ax.plot(
            group[date_col],
            group[value_col],
            marker=marker,
            linewidth=linewidth,
            label=site_name,
        )

    ax.set_xlabel("Date")
    ax.set_ylabel(value_col)
    ax.set_title(f"{value_col} Time Series by Site")
    ax.legend(title="Site")
    ax.grid(True)
    plt.xticks(rotation=45)
    plt.tight_layout()

    return ax

In [15]:
ndvi_vis = {
    'min': -0.2,
    'max': 0.8,
    'palette': [
        '#a50026', '#d73027', '#f46d43', '#fdae61',
        '#fee08b', '#d9ef8b', '#a6d96a', '#66bd63',
        '#1a9850', '#006837'
    ]
}

rgb_vis = {
    "bands": ["B4", "B3", "B2"],  # Red, Green, Blue
    "min": 0.0,
    "max": 0.3,
}

In [16]:
Map.addLayer(buda_col, rgb_vis, 'Buda')
Map.centerObject(sites_geom)
Map

Map(bottom=537006.0000121307, center=[-4.362158542390503, 39.41722881221705], controls=(WidgetControl(options=…